In [1]:
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

import sys
sys.path.append("../")

/Users/ohi/Documents/GitHub/NanoAgent/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Qwen 3.5 0.8b: https://github.com/rasbt/LLMs-from-scratch/tree/main/ch05/16_qwen3.5

model_name = "Qwen/Qwen3.5-0.8B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    low_cpu_mem_usage=True,
    dtype=torch.bfloat16
)

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 320/320 [00:00<00:00, 1214.73it/s]


In [3]:
print(model)

Qwen3_5ForCausalLM(
  (model): Qwen3_5TextModel(
    (embed_tokens): Embedding(248320, 1024)
    (layers): ModuleList(
      (0-2): 3 x Qwen3_5DecoderLayer(
        (linear_attn): Qwen3_5GatedDeltaNet(
          (act): SiLUActivation()
          (conv1d): Conv1d(6144, 6144, kernel_size=(4,), stride=(1,), padding=(3,), groups=6144, bias=False)
          (norm): Qwen3_5RMSNormGated()
          (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (in_proj_qkv): Linear(in_features=1024, out_features=6144, bias=False)
          (in_proj_z): Linear(in_features=1024, out_features=2048, bias=False)
          (in_proj_b): Linear(in_features=1024, out_features=16, bias=False)
          (in_proj_a): Linear(in_features=1024, out_features=16, bias=False)
        )
        (mlp): Qwen3_5MLP(
          (gate_proj): Linear(in_features=1024, out_features=3584, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3584, bias=False)
          (down_proj): Linear(

In [4]:
# for name, param in model.named_parameters():
#     print(name, param.size())

params = []

for idx, (name, module) in enumerate(model.named_modules()):
    if name in ['model', 'model.embed_tokens']: continue
    if len(name.split('.')) != 3:
        continue
    pytorch_total_params = sum(p.numel() for p in module.parameters())
    print(name, ':', pytorch_total_params)
    params.append(pytorch_total_params)
    if len(params) == 12: break

    print("-"*20)

print("TOTAL:", sum(params)/1e6)
    

model.layers.0 : 21555360
--------------------
model.layers.1 : 21555360
--------------------
model.layers.2 : 21555360
--------------------
model.layers.3 : 18352640
--------------------
model.layers.4 : 21555360
--------------------
model.layers.5 : 21555360
--------------------
model.layers.6 : 21555360
--------------------
model.layers.7 : 18352640
--------------------
model.layers.8 : 21555360
--------------------
model.layers.9 : 21555360
--------------------
model.layers.10 : 21555360
--------------------
model.layers.11 : 18352640
TOTAL: 249.05616


In [5]:
for name, param in model.named_parameters():
    print(name, param.size(), param.shape, param.data.shape)

model.embed_tokens.weight torch.Size([248320, 1024]) torch.Size([248320, 1024]) torch.Size([248320, 1024])
model.layers.0.linear_attn.dt_bias torch.Size([16]) torch.Size([16]) torch.Size([16])
model.layers.0.linear_attn.A_log torch.Size([16]) torch.Size([16]) torch.Size([16])
model.layers.0.linear_attn.conv1d.weight torch.Size([6144, 1, 4]) torch.Size([6144, 1, 4]) torch.Size([6144, 1, 4])
model.layers.0.linear_attn.norm.weight torch.Size([128]) torch.Size([128]) torch.Size([128])
model.layers.0.linear_attn.out_proj.weight torch.Size([1024, 2048]) torch.Size([1024, 2048]) torch.Size([1024, 2048])
model.layers.0.linear_attn.in_proj_qkv.weight torch.Size([6144, 1024]) torch.Size([6144, 1024]) torch.Size([6144, 1024])
model.layers.0.linear_attn.in_proj_z.weight torch.Size([2048, 1024]) torch.Size([2048, 1024]) torch.Size([2048, 1024])
model.layers.0.linear_attn.in_proj_b.weight torch.Size([16, 1024]) torch.Size([16, 1024]) torch.Size([16, 1024])
model.layers.0.linear_attn.in_proj_a.weight

In [ ]:
model.model.embed_tokens.weight.data = model.model.embed_tokens.weight.data[:1024, ...]

tensor([[ 0.0137,  0.0133, -0.0178,  ..., -0.0035, -0.0320, -0.0123],
        [ 0.0060,  0.0272,  0.0011,  ..., -0.0145, -0.0312,  0.0204],
        [ 0.0035, -0.0150,  0.0135,  ...,  0.0121, -0.0214, -0.0188],
        ...,
        [ 0.0247,  0.0018, -0.0146,  ..., -0.0071,  0.0195, -0.0073],
        [ 0.0247,  0.0018, -0.0146,  ..., -0.0071,  0.0195, -0.0073],
        [ 0.0247,  0.0018, -0.0146,  ..., -0.0071,  0.0195, -0.0073]],
       dtype=torch.bfloat16)

In [9]:
model.model.embed_tokens.weight.data.shape

torch.Size([248320, 1024])